# Buổi 5 — Tự viết notebook TF-IDF Text Retrieval

**Đích hôm nay:** từ bảng dữ liệu, tạo vector tài liệu, đưa câu hỏi vào đúng hệ cột, xếp hạng cosine và trả lại top-k văn bản.

Notebook gốc trong b2 dùng file vi_text_retrieval.csv. File CSV đó chưa có trong thư mục, vì vậy bài dùng DataFrame mẫu tự chứa để bạn vẫn học và chạy được. Khi tìm được CSV, phần cuối hướng dẫn thay nguồn dữ liệu.

Chạy từng cell từ trên xuống. Trước mỗi đoạn, tự nói: dữ liệu đầu vào là gì và kết quả muốn nhận là gì?

## 1. Chuẩn bị thư viện

pandas tạo và đọc bảng. scikit-learn tạo TF-IDF và tính cosine. Nếu chạy Colab/Jupyter mà báo thiếu thư viện, bỏ dấu # ở lệnh pip, chạy xong rồi chạy lại import.

In [ ]:
# Nếu thiếu thư viện, bỏ dấu # ở dòng dưới rồi chạy một lần:
# %pip install pandas scikit-learn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Tạo bảng tài liệu

DataFrame giống bảng tính: một hàng là một ví dụ; cột text giữ đoạn cần tìm; cột question có câu hỏi mẫu. Các list truyền vào DataFrame phải có cùng số dòng.

In [ ]:
vi_data_df = pd.DataFrame({
    "text": [
        "TF-IDF giảm trọng số của từ phổ biến.",
        "Cosine so sánh hướng của hai vector.",
        "Vocabulary quyết định các cột trong vector.",
        "Python dùng hàm để gom các bước xử lý."
    ],
    "question": [
        "Từ phổ biến bị xử lý thế nào?",
        "Làm sao so hai vector gần nhau?",
        "Mỗi vị trí số biểu diễn điều gì?",
        "Hàm trong Python có tác dụng gì?"
    ]
})

print(vi_data_df)
print("Các cột:", list(vi_data_df.columns))

Chọn cột text rồi đổi sang list Python. Chữ thường giúp "Cosine" và "cosine" cùng dạng. Sau này câu hỏi cũng cần lower giống tài liệu.

In [ ]:
context = vi_data_df["text"].tolist()
context = [doc.lower() for doc in context]

print("Số tài liệu:", len(context))
print(context[0])

## 3. fit_transform: học cột và mã hóa tài liệu

TfidfVectorizer làm TF-IDF cho ta. fit học vocabulary và trọng số IDF từ tập tài liệu; transform biến các câu thành vector. fit_transform làm cả hai việc trên context.

Ma trận thường là sparse: nó chỉ lưu những vị trí khác 0 để dùng ít bộ nhớ hơn.

In [ ]:
vectorizer = TfidfVectorizer()
doc_matrix = vectorizer.fit_transform(context)

print("Số tài liệu, số cột:", doc_matrix.shape)
print("Một số từ đã học:", vectorizer.get_feature_names_out()[:10])

Nếu N là số tài liệu, shape bắt đầu bằng N. Số cột là số từ vectorizer học được sau khi bỏ qua token không dùng được. Mỗi hàng của doc_matrix ứng với đúng một phần tử trong context.

## 4. Biến câu hỏi thành vector

Câu hỏi phải dùng chính vectorizer đã fit trên context. Không fit riêng query: nếu vocabulary khác, cột số không còn cùng nghĩa.

Bọc question trong list vì vectorizer nhận danh sách các câu, kể cả khi chỉ có một câu.

In [ ]:
question = vi_data_df.iloc[0]["question"].lower()
query_vector = vectorizer.transform([question])

scores = cosine_similarity(query_vector, doc_matrix).ravel()
print("Question:", question)
print("Shape query:", query_vector.shape)
print("Shape điểm:", scores.shape)
print(scores.round(3))

iloc[0] lấy hàng đầu tiên; ["question"] lấy nội dung cột question. cosine_similarity so một query với mọi hàng tài liệu và tạo điểm. ravel đổi kết quả từ dạng một hàng thành list điểm dài N.

Nếu điểm nào đó bằng 0, nghĩa là không có hướng/từ chung sau vector hóa; vị trí vẫn có thể xuất hiện trong top-k nhưng không nên coi là câu trả lời.

## 5. Viết corr_search và trả về top-k

argsort trả các chỉ số theo điểm tăng dần. [::-1] đảo thứ tự để điểm lớn đứng trước. [:top_k] chỉ giữ k chỉ số đầu. Sau đó dùng cùng idx để lấy điểm và đoạn văn, tránh ghép nhầm.

In [ ]:
def corr_search(question, top_k=3):
    query_vector = vectorizer.transform([question.lower()])
    scores = cosine_similarity(query_vector, doc_matrix).ravel()
    top_ids = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_ids:
        results.append({
            "id": int(idx),
            "cosine_score": float(scores[idx]),
            "text": context[idx]
        })
    return results

results = corr_search("Từ phổ biến bị xử lý thế nào?", top_k=2)
for result in results:
    print(result["id"], round(result["cosine_score"], 3))
    print(result["text"])
    print()

Tên corr_search được giữ để bạn nhận ra hàm trong notebook gốc. Dù tên gọi có chữ corr, code dùng cosine_similarity; đây không phải phép tính correlation Pearson. Nên đọc cách tính thật bên trong hàm.

## 6. Tự thử truy vấn và đọc kết quả

Hãy hỏi về cosine, vocabulary hoặc Python. Xem ID có trỏ đúng văn bản trong context không. Thử top_k lớn hơn số tài liệu: chương trình chỉ trả số tài liệu có thật.

In [ ]:
# TỰ LÀM: sửa câu hỏi và k. Thử cả một câu có từ đã học
# và một câu chỉ có từ lạ.
my_question = "Vocabulary tạo các cột như thế nào?"
my_results = corr_search(my_question, top_k=3)
for item in my_results:
    print(item["id"], round(item["cosine_score"], 3), item["text"])

## 7. Khi có CSV thật

CSV vi_text_retrieval.csv chưa có trong thư mục b2. Nếu sau này có file đặt cạnh notebook, chạy các dòng dưới và xem tên cột. Cần thay "text" / "question" nếu CSV dùng tên khác.

Chú ý: sau khi nạp CSV, phải tạo lại context, fit vectorizer trên context mới và dùng hàm corr_search với các biến mới.

In [ ]:
# Khi đã có file CSV bên cạnh notebook, bỏ dấu #:
# vi_data_df = pd.read_csv("vi_text_retrieval.csv")
# print(vi_data_df.columns)
# print(vi_data_df.head())

# Nếu cột văn bản trong CSV có tên khác, đổi "text" ở dòng này:
# context = vi_data_df["text"].astype(str).str.lower().tolist()
# vectorizer = TfidfVectorizer()
# doc_matrix = vectorizer.fit_transform(context)

## 8. Bài kiểm tra notebook trắng

Che các cell lời giải hoặc mở notebook mới. Tự dựng lại:
1. DataFrame và context.
2. TfidfVectorizer, fit_transform trên context.
3. query_vector bằng transform của vectorizer đã fit.
4. cosine scores và ravel.
5. argsort giảm dần, cắt top-k.
6. Hàm corr_search trả ID, score và text.
7. Chạy với hai câu hỏi mới.

### Tự chấm
Bạn đạt khi code chạy trên dữ liệu khác ví dụ, giải thích được vì sao query không fit riêng, và chỉ ra score nào thuộc văn bản nào.

## Lời giải và liên hệ với file gốc

Toàn bộ pipeline mẫu nằm trong các cell phía trên theo thứ tự nhỏ đến lớn. Sau khi tự làm, đối chiếu từng bước:
- DataFrame → context.
- fit_transform(context) học không gian từ.
- transform([question]) dùng lại không gian ấy.
- cosine_similarity cho điểm.
- argsort lấy vị trí top-k.
- ID tra lại context để in nội dung.

Hàm corr_search ở notebook gốc còn dở; bài này đã hoàn thiện phần đó. Ngày thứ ba của lộ trình nhằm chuẩn bị để bạn tự viết lại notebook retrieval, không phải chỉ bấm chạy code có sẵn.